# Atividade Prática 11 - Lidando com Dados do Mundo Real

Este _notebook_ é responsável por acessar as páginas web da kworb e salvar as informações das 200 músicas mais populares globalmente deste ano `.csv`. 

Para isso se baixa as informações dos HTMLs utilizando Pandas.


In [1]:
import pandas as pd

df = pd.read_html('https://kworb.net/ww/archive/20260820.html') # Pelo HTML se tratar de dados tabulares se torna mais 
                                                                # simples a extração.
df = df[0] # ww significa world wide (mundialmente)

In [2]:
df.head()

,Pos,P+,Artist and Title,Days,Pk,(x?),Pts,Pts+,TPts,US,...,LT,NG,PE,PT,RO,SA,SI,SK,UA,VN
0,1,+1,BIGBANG - BiiiG,2,1,(x1),16190,967,0.031,13.0,...,NaN,NaN,1.0,NaN,83.0,1.0,NaN,48.0,89.0,1.0
1,2,-1,Shakira & Burna Boy - Dai Dai,98,1,(x40),15474,-351,1.497,16.0,...,NaN,NaN,NaN,NaN,9.0,4.0,2.0,1.0,NaN,NaN
2,3,=,"HUGEL, Imael Angel & Ultra Naté - Movin' To Th...",91,1,(x2),15000,35,1.130,52.0,...,79.0,NaN,110.0,NaN,5.0,NaN,NaN,2.0,NaN,NaN
3,4,+1,Ella Langley - Choosin' Texas,308,4,NaN,9548,-85,1.789,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,-1,Kimo Sounds - Rise Slow,80,3,(x8),9044,-1078,0.583,99.0,...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,50.0,99.0,NaN


In [3]:

df.info() #

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 69 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Pos               200 non-null    int64  
 1   P+                200 non-null    str    
 2   Artist and Title  200 non-null    str    
 3   Days              200 non-null    int64  
 4   Pk                200 non-null    int64  
 5   (x?)              47 non-null     str    
 6   Pts               200 non-null    int64  
 7   Pts+              200 non-null    int64  
 8   TPts              200 non-null    float64
 9   US                97 non-null     float64
 10  UK                95 non-null     float64
 11  DE                84 non-null     float64
 12  AU                91 non-null     float64
 13  JP                31 non-null     float64
 14  AT                47 non-null     float64
 15  BE                43 non-null     float64
 16  CA                94 non-null     float64
 17  CH      

In [4]:
ww_df = df[['Artist and Title', 'Pts', 'Pts+', 'TPts']] # Informações que serão utilizadas.
ww_df.head()

,Artist and Title,Pts,Pts+,TPts
0,BIGBANG - BiiiG,16190,967,0.031
1,Shakira & Burna Boy - Dai Dai,15474,-351,1.497
2,"HUGEL, Imael Angel & Ultra Naté - Movin' To Th...",15000,35,1.130
3,Ella Langley - Choosin' Texas,9548,-85,1.789
4,Kimo Sounds - Rise Slow,9044,-1078,0.583


Será criado um DataFrame com o Nome do Artista (_artista_), Nome da Música (_musica_), média de pontos (_pts_mean_), média de pontos ganhos(_pts_gan_mean_), média do total de pontos(_pts_tot_mean_),

Primeiro foram reunidas as informações diárias dos ranques.

In [5]:
# Teste da função utilizada para todos os demais htmls
def extrair_artistas(artistas_e_musica):
    try :
        
        artistas, musica = artistas_e_musica.strip().rsplit(' - ', 1)# Utiliza o `rsplit` que lê a linha da direita 
                                                    # para a esquerda com no máximo uma ocorrência, como o nome da 
                                                    # música vem antes nessa ordem, logo é separado
    
        artistas = artistas.strip().split(', ') # Artistas são separados por `,` ou por `&` 
                                                    
        artistas = [a for a_s in artistas for a in a_s.strip().rsplit(' & ', 1)] # Basicamente um `flat map`
                                                                                # Pois ao serparar por `,` depois se deve
                                                                                # separar por `&` utiliando `a_s`, caso
                                                                                # não haja será  finaliza, se for percorre 
                                                                                # a nova lista `a`, retornando uma lista.
                                                                                # Também lido da direita para a esquerda.
        
        return artistas, musica # Retorna os artistas da música e a música em uma tupla

    except Exception as e:
        print(artistas_e_musica)
    raise e

def extrair_info(linha):
    
    artistas, musica = extrair_artistas(linha['Artist and Title']) 
    musicas = [] # Extraída os artistas da música demais informações são distribuídas por referenciar ao mesmo artista.
    for artista in artistas:
        musicas.append([artista, musica, linha['Pts'], linha['Pts+'], linha['TPts']])

    return musicas

def montar_dataframe(dataframe):
    infos_lista = dataframe.apply(extrair_info, axis=1) # Para percorrer por linha e não coluna usa-se `axis`=1
    
    infos = [info for infos in infos_lista for info in infos] # Novamente list comprehension para pegar a lista de listas
                                                                # e planar elas (`flat`)
    
    tabela_musica = pd.DataFrame(data=infos, columns=['artista', 'musica', 'pontos', 'pts_ganho', 'pts_calc'])
                                                    # Colunas definas, utilizadas para os cálculos posteriores
    
    return tabela_musica

montar_dataframe(ww_df)

,artista,musica,pontos,pts_ganho,pts_calc
0,BIGBANG,BiiiG,16190,967,0.031
1,Shakira,Dai Dai,15474,-351,1.497
2,Burna Boy,Dai Dai,15474,-351,1.497
3,HUGEL,Movin' To The Sun,15000,35,1.130
4,Imael Angel,Movin' To The Sun,15000,35,1.130
...,...,...,...,...,...
245,Eminem,Lose Yourself,1044,94,3.860
246,Usagi(CV:Ozawa Ari),Usagi Rap,1041,114,0.031
247,Momonga(CV:Iguchi Yuka),Usagi Rap,1041,114,0.031
248,Mermaid(CV:Oki Saeko),Usagi Rap,1041,114,0.031


In [6]:
import datetime as dt
import time

def data_to_str(data):
    '''Converte a data para o formato utilizado na URL. Ex: para a data do dia 20/08/2026 retorna "20260820"'''

    year = data.year
    
    month_str = str(data.month) 
    if data.month < 10:
        month_str = f'0{str(data.month)}' # O formato utilizado coloca 0 em numeoros sem casas decimais ou seja ao 
                                            # invés de 8 é utilizado '08'. Essa regra se aplica para meses e dias.

    day_str = str(data.day) 
    if data.day < 10:
        day_str = f'0{str(data.day)}' # Convertido conforme formato explicado.
        
    
    return f'{year}{month_str}{day_str}'
    

def extrair_tabela_dia(data):
    '''Retorna um `DataFrame` com as informações diárias das músicas'''

    str_date = data_to_str(data)

    url = f'https://kworb.net/ww/archive/{str_date}.html' # Como cada endereço do site possui apenas a data de
                                                            # diferença basta mudar a data da url

    print(f'Extraindo dados da url... \n{url}')

    time.sleep(1) # Temporizador para não enviar muitas requisições de uma vez para o site. 
    
    df_data = pd.read_html(url)[0] # Acessa o DataFrame da página semelhante ao teste no início do notebook.

    print(f'Montando `DataFrame`...')
    
    df_montado = montar_dataframe(df_data) # Monta o `DataFrame` conforme o teste.

    df_montado['data'] = str_date # Para que possa indentificar o dia dos registros das datas.

    print(f'Retornando novo `DataFrame`\n')
    
    return df_montado


# data_exemplo = dt.date.fromisoformat('2026-08-19')
data_exemplo = dt.date.fromisoformat('2026-01-10')

df_dia = extrair_tabela_dia(data_exemplo)

nome_arquivo = f'kworb_ww_{data_to_str(data_exemplo)}.csv'
df_dia.to_csv('dados/teste_'+nome_arquivo, sep=';', decimal=',', index=False) # Só vai ser escrito se todos os meses funcionarem

Extraindo dados da url... 
https://kworb.net/ww/archive/20260110.html
Montando `DataFrame`...
Retornando novo `DataFrame`



Foi percorrido as páginas desde o início do ano até o dia 20 de Agosto de 2026, a iteração é feita em intervalos de meses e cada mês é salvo para que haja um ponto de 
parada de cada mês caso haja algum erro e precise ser tratado.

In [7]:
def intervalo_data(data_inicio, data_fim):
    '''Cria um intervalo com base nas datas informadas'''
    return (dt.date.fromisoformat(data_inicio), dt.date.fromisoformat(data_fim))

janeiro = intervalo_data('2026-01-01', '2026-01-31')
fevereiro = intervalo_data('2026-02-01', '2026-02-28')
marco = intervalo_data('2026-03-01', '2026-03-31')
abril = intervalo_data('2026-04-01', '2026-04-30')
maio = intervalo_data('2026-05-01', '2026-05-31')
junho = intervalo_data('2026-06-01', '2026-06-30')
julho = intervalo_data('2026-07-01', '2026-07-31')
agosto = intervalo_data('2026-08-01', '2026-08-20')

meses = [janeiro, fevereiro, marco, abril, junho, julho, agosto] # Intervalos definidos

colunas = ['artista', 'musica', 'pontos', 'pts_ganho', 'pts_calc', 'data']

df_ano = pd.DataFrame(data=[], columns=colunas)
for mes in meses:
    data_inicio, data_fim = mes

    print(f'\nIniciado o mês com o intervalo: {data_inicio} | {data_fim}')

    df_mes = pd.DataFrame(data=[], columns=colunas)
    
    data_atual = data_inicio
    while True:
        print(f'Iniciando extração do dia {data_atual}')
        
        df_dia = extrair_tabela_dia(data_atual)

        df_mes = pd.concat([df_mes, df_dia]) # Monta o `DataFrame` do mês de acordo com cada dia

        if data_atual != data_fim: # Condição de parada do laço
            data_atual = data_atual.replace(day = data_atual.day +1) # Adiciona um dia na Data (iteração)
        else :
            break 

    nome_arquivo = f'kworb_ww_{data_to_str(data_inicio)}-{data_to_str(data_fim)}.csv'

    print(f'Salvando arquivo {nome_arquivo}...')
    df_mes.to_csv('dados/'+nome_arquivo, sep=';', decimal=',', index=False) 

    df_ano = pd.concat([df_ano, df_mes]) # Monta o `DataFrame` do ano de acordo com cada mês
    
    print('Mês finalizado')

df_ano.to_csv('dados/kworb_ww_2026.csv', sep=';', decimal=',', index=False) # Só vai ser escrito se todos os meses funcionarem


Iniciado o mês com o intervalo: 2026-01-01 | 2026-01-31
Iniciando extração do dia 2026-01-01
Extraindo dados da url... 
https://kworb.net/ww/archive/20260101.html
Montando `DataFrame`...
Retornando novo `DataFrame`

Iniciando extração do dia 2026-01-02
Extraindo dados da url... 
https://kworb.net/ww/archive/20260102.html
Montando `DataFrame`...
Retornando novo `DataFrame`

Iniciando extração do dia 2026-01-03
Extraindo dados da url... 
https://kworb.net/ww/archive/20260103.html
Montando `DataFrame`...
Retornando novo `DataFrame`

Iniciando extração do dia 2026-01-04
Extraindo dados da url... 
https://kworb.net/ww/archive/20260104.html
Montando `DataFrame`...
Retornando novo `DataFrame`

Iniciando extração do dia 2026-01-05
Extraindo dados da url... 
https://kworb.net/ww/archive/20260105.html
Montando `DataFrame`...
Retornando novo `DataFrame`

Iniciando extração do dia 2026-01-06
Extraindo dados da url... 
https://kworb.net/ww/archive/20260106.html
Montando `DataFrame`...
Retornando n

### Referências

[1] https://kworb.net/ww/ (20/08/2026).

[2] https://docs.python.org/3/library/datetime.html (20/08/2026).

[3] https://stackoverflow.com/questions/15012228/splitting-on-last-delimiter-in-python-string (20/08/2026)